In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import ( col, when, trim, lit, regexp_replace, split, explode, lower, array_contains, transform, isnan, count, round)
from pyspark.sql.types import IntegerType
from delta import *

warehouse_location = 'hdfs://hdfs-nn:9000/projeto/silver'

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("ETL-Netflix-Credits-Silver") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada com sucesso e Delta Lake configurado.")

SparkSession iniciada com sucesso e Delta Lake configurado.


In [2]:
hdfs_path = "hdfs://hdfs-nn:9000/projeto/bronze/netflix_credits.csv"

In [3]:
df_bronze = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True,
    quote='\"',
    escape='\"',
    multiLine=True
)

In [4]:
print(f"Total de registos lidos da camada Bronze: {df_bronze.count()}")
df_bronze.printSchema()
df_bronze.toPandas()

Total de registos lidos da camada Bronze: 77801
root
 |-- person_id: integer (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- character: string (nullable = true)
 |-- role: string (nullable = true)



,person_id,id,name,character,role
0,3748,tm84618,Robert De Niro,Travis Bickle,ACTOR
1,14658,tm84618,Jodie Foster,Iris Steensma,ACTOR
2,7064,tm84618,Albert Brooks,Tom,ACTOR
3,3739,tm84618,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,48933,tm84618,Cybill Shepherd,Betsy,ACTOR
...,...,...,...,...,...
77796,736339,tm1059008,Adelaida Buscato,María Paz,ACTOR
77797,399499,tm1059008,Luz Stella Luengas,Karen Bayona,ACTOR
77798,373198,tm1059008,Inés Prieto,Fanny,ACTOR
77799,378132,tm1059008,Isabel Gaona,Cacica,ACTOR


In [5]:
from pyspark.sql.functions import col, trim, count

print(f"Total de registos no 'df_bronze': {df_bronze.count()}")

# Procura por linhas onde 'character' é nulo ou uma string vazia (após trim)
character_vazios = df_bronze.filter(
    (col("character").isNull()) | (trim(col("character")) == "")
)

print(f"Total de 'character' nulos/vazios encontrados: {character_vazios.count()}")

# Mostra os 'role' distintos onde 'character' é nulo
character_vazios.select("role").distinct().show()

# Mostra uma amostra
character_vazios.select("id", "name", "character", "role").show(5)

Total de registos no 'df_bronze': 77801
Total de 'character' nulos/vazios encontrados: 9772
+--------+
|    role|
+--------+
|DIRECTOR|
|   ACTOR|
+--------+

+--------+---------------+---------+--------+
|      id|           name|character|    role|
+--------+---------------+---------+--------+
| tm84618|Martin Scorsese|     null|DIRECTOR|
|tm154986|   John Boorman|     null|DIRECTOR|
|tm127384|    Terry Jones|     null|DIRECTOR|
|tm127384|  Terry Gilliam|     null|DIRECTOR|
|tm120801| Robert Aldrich|     null|DIRECTOR|
+--------+---------------+---------+--------+
only showing top 5 rows



In [6]:
from pyspark.sql.functions import lit

# Cria o novo DataFrame 'df_character' com a coluna corrigida
df_character = df_bronze.withColumn("character",
    when(
        (col("character").isNull()) | (trim(col("character")) == ""),
        lit("NA")  # Coloca a string literal "NA"
    ).otherwise(col("character")) # Mantém o valor original
)

In [7]:
character_vazios_depois = df_character.filter(
    (col("character").isNull()) | (trim(col("character")) == "")
)
num_vazias = character_vazios_depois.count()

if num_vazias == 0:
    print("✅ SUCESSO: Não foram encontrados mais 'character' nulos/vazios.")
else:
    print(f"❌ FALHA: Ainda existem {num_vazias} 'character' nulos/vazios.")
    character_vazios_depois.show()

character_na = df_character.filter(
    col("character") == "NA"
)
count_na = character_na.count()

print(f"Total de registos com character = 'NA': {count_na}")

character_na.select("role").distinct().show()

✅ SUCESSO: Não foram encontrados mais 'character' nulos/vazios.
Total de registos com character = 'NA': 9772
+--------+
|    role|
+--------+
|DIRECTOR|
|   ACTOR|
+--------+



In [8]:
from pyspark.sql.functions import lower, trim, col, translate, regexp_replace

# Colunas de texto a limpar
text_columns_to_clean = ["name", "character", "role"]

# O nosso DataFrame de trabalho
df_final_limpo = df_character

# Mapa de acentos
accent_map = {
    'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
    'ã': 'a', 'õ': 'o', 'â': 'a', 'ê': 'e', 'ô': 'o', 'ç': 'c',
    'ñ': 'n', 'ü': 'u', 'ë': 'e', 'ï': 'i', 'ö': 'o',
    'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
    'Ã': 'A', 'Õ': 'O', 'Â': 'A', 'Ê': 'E', 'Ô': 'O', 'Ç': 'C',
    'Ñ': 'N', 'Ü': 'U', 'Ë': 'E', 'Ï': 'I', 'Ö': 'O'
}

# Converte o mapa em duas strings para a função translate()
# Esta é a forma mais eficiente de o fazer em Spark
from_chars = "".join(accent_map.keys())
to_chars = "".join(accent_map.values())


for col_name in text_columns_to_clean:

# Começa com a coluna original
    cleaned_col = col(col_name)

    # Remove espaços no início/fim (Trim)
    cleaned_col = trim(cleaned_col)

    # Remove acentos (Diacritic Folding)
    # Usamos translate() que é muito mais rápido que múltiplos regexp_replace
    cleaned_col = translate(cleaned_col, from_chars, to_chars)

    # Converte para minúsculo (Lowercase)
    cleaned_col = lower(cleaned_col)

    # Remover espaços duplos
    cleaned_col = regexp_replace(cleaned_col, " +", " ")

    # Atualiza o DataFrame com a coluna limpa
    df_final_limpo = df_final_limpo.withColumn(col_name, cleaned_col)

# Mostra o resultado
df_final_limpo.select(text_columns_to_clean).show(truncate=False)

+------------------+-----------------------------+-----+
|name              |character                    |role |
+------------------+-----------------------------+-----+
|robert de niro    |travis bickle                |actor|
|jodie foster      |iris steensma                |actor|
|albert brooks     |tom                          |actor|
|harvey keitel     |matthew 'sport' higgins      |actor|
|cybill shepherd   |betsy                        |actor|
|peter boyle       |wizard                       |actor|
|leonard harris    |senator charles palantine    |actor|
|diahnne abbott    |concession girl              |actor|
|gino ardito       |policeman at rally           |actor|
|martin scorsese   |passenger watching silhouette|actor|
|murray moston     |iris' time keeper            |actor|
|richard higgs     |secret service agent         |actor|
|bill minkin       |tom's assistant (uncredited) |actor|
|bob maroff        |mafioso (uncredited)         |actor|
|victor argo       |melio, deli

In [9]:
df_final_limpo \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("hdfs://hdfs-nn:9000/projeto/silver/netflix_credits")

In [10]:
df_final_limpo.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("role") \
    .option("overwriteSchema", "true") \
    .option("path", "hdfs://hdfs-nn:9000/warehouse/projeto.db/netflix_credits") \
    .saveAsTable("projeto.netflix_credits")

In [11]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.netflix_credits
    """
).show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2025-11-17 16:42:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|          0|  Serializable|        false|{numFiles -> 2, n...|        null|Apache-Spark/3.4....|
|      0|2025-11-16 23:40:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|       null|  Serializable|        false|{numFiles -

In [12]:
spark.sql(
    """
    SELECT * FROM projeto.netflix_credits
    """
).show()

+---------+-------+------------------+--------------------+-----+
|person_id|     id|              name|           character| role|
+---------+-------+------------------+--------------------+-----+
|     3748|tm84618|    robert de niro|       travis bickle|actor|
|    14658|tm84618|      jodie foster|       iris steensma|actor|
|     7064|tm84618|     albert brooks|                 tom|actor|
|     3739|tm84618|     harvey keitel|matthew 'sport' h...|actor|
|    48933|tm84618|   cybill shepherd|               betsy|actor|
|    32267|tm84618|       peter boyle|              wizard|actor|
|   519612|tm84618|    leonard harris|senator charles p...|actor|
|    29068|tm84618|    diahnne abbott|     concession girl|actor|
|   519613|tm84618|       gino ardito|  policeman at rally|actor|
|     3308|tm84618|   martin scorsese|passenger watchin...|actor|
|    43791|tm84618|     murray moston|   iris' time keeper|actor|
|   519614|tm84618|     richard higgs|secret service agent|actor|
|   519615

In [13]:
spark.stop()